# Maintenance Work Order Analysis with Ollama

This notebook reads the sample maintenance work order CSV, calls a local Ollama model, and extracts the likely equipment and failure mode from each work order description.

Recommended starting model for a Dell Inspiron 15 5510: `qwen2.5:3b`. It is small enough for typical CPU-only laptop use while still being capable at structured extraction. If the laptop has 16 GB RAM and you can tolerate slower runs, try `qwen2.5:7b`. If it has 8 GB RAM or feels sluggish, use `llama3.2:3b` or `gemma3:1b`.

## One-time setup

Install Ollama, then pull the model in a terminal:

```bash
ollama pull qwen2.5:3b
ollama serve
```

Install notebook dependencies if needed:

```bash
python3 -m pip install -r requirements.txt
```

In [1]:
print("Loading standard library imports...")
from pathlib import Path
import json
import os
import re

print("Loading requests...")
import requests

print("Loading pandas...")
import pandas as pd

print("Imports loaded successfully")

Loading standard library imports...
Loading requests...
Loading pandas...
Imports loaded successfully


In [2]:
CSV_PATH = Path("../data/maintenance_work_orders.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("data/maintenance_work_orders.csv")

OUTPUT_PATH = CSV_PATH.with_name("maintenance_work_orders_analysed.csv")

OLLAMA_BASE_URL = "http://localhost:11434"
MODEL = os.environ.get("OLLAMA_MODEL", "qwen2.5:3b")

df = pd.read_csv(CSV_PATH)
df.head()

,work_order_number,description,date_created,breakdown_type,work_order_status
0,WO-2026-0001,Conveyor CV-101 stopped during packaging line ...,2026-01-06,breakdown,Completed
1,WO-2026-0002,Monthly preventive inspection of centrifugal p...,2026-01-08,preventive,Completed
2,WO-2026-0003,Air compressor AC-02 failed to maintain plant ...,2026-01-10,corrective,In Progress
3,WO-2026-0004,Mixer MX-310 making intermittent grinding nois...,2026-01-14,corrective,Open
4,WO-2026-0005,Replace worn photoelectric sensor on palletize...,2026-01-17,corrective,Completed


In [3]:
def get_ollama_models() -> list[str]:
    try:
        response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError("Ollama is not reachable. Start it with `ollama serve`.") from exc

    return [model["name"] for model in response.json().get("models", [])]


def check_ollama_available() -> None:
    installed_models = get_ollama_models()
    if MODEL not in installed_models:
        available = ", ".join(installed_models) if installed_models else "none"
        raise RuntimeError(
            f"Ollama is running, but model '{MODEL}' is not installed. "
            f"Run `ollama pull {MODEL}` in a terminal, or set MODEL to one of: {available}."
        )


check_ollama_available()

RuntimeError: Ollama is not reachable. Start it with `ollama serve` and make sure the model is pulled.

In [ ]:
SYSTEM_PROMPT = """
You are a maintenance reliability analyst. Extract structured failure information from work order text.

Return only valid JSON with these exact keys:
- equipment_failed: the specific asset or equipment mentioned, such as pump P-204 or conveyor CV-101.
- failure_mode: concise physical or functional failure mode, such as mechanical seal leak, motor overload trip, bearing wear, sensor false trigger, blocked suction strainer, no failure - preventive maintenance.
- confidence: high, medium, or low.
- evidence: short phrase from the work order that supports the answer.

Rules:
- If breakdown_type is preventive and the description does not report a fault, set failure_mode to "no failure - preventive maintenance".
- Do not invent equipment that is not present in the description.
- Keep each value short and practical for maintenance analysis.
""".strip()


def build_user_prompt(row: pd.Series) -> str:
    return f"""
Work order number: {row.work_order_number}
Date created: {row.date_created}
Breakdown type: {row.breakdown_type}
Status: {row.work_order_status}
Description: {row.description}
""".strip()

In [ ]:
def parse_json_response(text: str) -> dict:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def raise_ollama_error(response: requests.Response) -> None:
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        detail = response.text.strip()
        if "not found" in detail.lower() and "model" in detail.lower():
            raise requests.HTTPError(
                f"Ollama model '{MODEL}' is not installed. Run `ollama pull {MODEL}` "
                "or update MODEL to an installed model. "
                f"Response body: {detail}"
            ) from exc
        raise requests.HTTPError(f"{exc}. Response body: {detail}") from exc


def call_ollama(row: pd.Series) -> str:
    payload = {
        "model": MODEL,
        "prompt": f"{SYSTEM_PROMPT}\n\n{build_user_prompt(row)}",
        "format": "json",
        "stream": False,
        "options": {"temperature": 0},
    }

    response = requests.post(f"{OLLAMA_BASE_URL}/api/generate", json=payload, timeout=120)
    raise_ollama_error(response)
    return response.json()["response"]


def analyse_work_order(row: pd.Series) -> dict:
    content = call_ollama(row)
    parsed = parse_json_response(content)

    return {
        "equipment_failed": parsed.get("equipment_failed", "unknown"),
        "failure_mode": parsed.get("failure_mode", "unknown"),
        "llm_confidence": parsed.get("confidence", "low"),
        "llm_evidence": parsed.get("evidence", ""),
    }

In [ ]:
sample_result = analyse_work_order(df.iloc[0])
sample_result

In [ ]:
results = []

for index, row in df.iterrows():
    print(f"Analysing {row.work_order_number} ({index + 1}/{len(df)})")
    try:
        result = analyse_work_order(row)
    except Exception as exc:
        result = {
            "equipment_failed": "error",
            "failure_mode": "error",
            "llm_confidence": "low",
            "llm_evidence": str(exc),
        }
    results.append(result)

llm_columns = ["equipment_failed", "failure_mode", "llm_confidence", "llm_evidence"]
llm_results_df = pd.DataFrame(results).reindex(columns=llm_columns)

analysis_df = pd.concat([df, llm_results_df], axis=1)
analysis_df.head(10)

In [ ]:
analysis_df.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH

In [ ]:
analysis_df.groupby(["breakdown_type", "failure_mode"]).size().reset_index(name="count").sort_values(
    ["breakdown_type", "count"], ascending=[True, False]
)